# cheaseBS at the campaign box edges

Does reshaping the profiles change how hard cheaseBS has to work?

Four solves per discharge: `Te_ped_scale` and `ne_ped_scale` at the frozen box
edges 0.7 and 1.3. Needs CHEASE, so this runs on NERSC.

**The open question.** The 2026-08-18 five-point 132543 run had four points
converging in 2 iterations and `ne` x1.3 needing 19, with no explanation. These
runs say whether that split follows the direction (up hard, down easy), the
variable (density hard, temperature easy), the discharge, or nothing
reproducible. The answer sets `max_iter` for the campaign, since a cap read off
the typical point truncates the hard one.

Each discharge gets its own cell: run, parameter/iteration table, then the
source gfile against its four reconstructions. **Start with 132543 and read the
output before running the rest.**

In [ ]:
%load_ext autoreload
%autoreload 2

import pathlib, sys

# Same root-walk the fit and scaling notebooks use, so this runs from any cwd.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pedestal_scan.py").exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "reshape_convergence"))

from pedestal_scan import Campaign
import reshape_helpers as rh

# Loads and fits all four baselines, so the fits under these solves are the
# same ones the fit and scaling notebooks scored.
camp = Campaign()
print("shots  ", camp.shots)
print("axes   ", rh.AXES)
print("scales ", rh.SCALES)
print("cheaseBS", rh.CHEASEBS)

## 132543 — test discharge

ELMy, low triangularity, and the shot the 2-vs-19 split came from.

Read before running the others:
- **`iters`** — does 1.30 cost more than 0.70? does `ne` cost more than `Te`?
- **`accepted`** — an `R` or a `RAISED` inside the frozen box is a campaign
  blocker, not a note.
- **`Ip_err`** — compare +delta against -delta on one axis. A one-sided response
  is the open asymmetry.
- **`dq_max` / `dq_at` / `dq_rms`** — the equilibrium change over the **whole**
  profile, and the radius where it peaks. This is the honest measure here:
  cheaseBS reshapes all of it, so a number read at two radii cannot say whether
  the reshape reached the equilibrium. `q_err@x0` is the separate question of
  whether the campaign's acceptance gate would take the point, and it is scored
  at the two GENE analysis radii by frozen policy (2026-08-16) — keep the two
  apart.
- **the plot** — a reconstruction lying on the black source curve means the
  reshape never reached the equilibrium.

In [ ]:
rows_132543, wd = rh.run_bounds(camp, 132543)
rh.table(rows_132543, shot=132543)

In [ ]:
# cheaseBS reshapes the whole equilibrium, so this is the primary view.
fig = rh.plot_gfiles(rows_132543, camp, 132543)

In [ ]:
# Pedestal zoom -- where the axes act. Secondary: a windowed view cannot
# show distortion outside the window.
fig = rh.plot_gfiles(rows_132543, camp, 132543, rho_tor=[0.7, 1.0])

## 132588 — ELM-free, high triangularity

Same radii as 132543. Its `ne` fit is pinned on the `b_pos` bound, so treat its `ne_ped_scale` rows as suspect until that is resolved.

In [ ]:
rows_132588, wd = rh.run_bounds(camp, 132588)
rh.table(rows_132588, shot=132588)

In [ ]:
# cheaseBS reshapes the whole equilibrium, so this is the primary view.
fig = rh.plot_gfiles(rows_132588, camp, 132588)

In [ ]:
# Pedestal zoom -- where the axes act. Secondary: a windowed view cannot
# show distortion outside the window.
fig = rh.plot_gfiles(rows_132588, camp, 132588, rho_tor=[0.7, 1.0])

## 129015 — ELMy, low triangularity

Single analysis radius 0.85.

In [ ]:
rows_129015, wd = rh.run_bounds(camp, 129015)
rh.table(rows_129015, shot=129015)

In [ ]:
# cheaseBS reshapes the whole equilibrium, so this is the primary view.
fig = rh.plot_gfiles(rows_129015, camp, 129015)

In [ ]:
# Pedestal zoom -- where the axes act. Secondary: a windowed view cannot
# show distortion outside the window.
fig = rh.plot_gfiles(rows_129015, camp, 129015, rho_tor=[0.7, 1.0])

## 129038 — ELM-free, low triangularity

Weakest fit of the four and a placeholder analysis radius, so its gate verdict carries less weight than the other three.

In [ ]:
rows_129038, wd = rh.run_bounds(camp, 129038)
rh.table(rows_129038, shot=129038)

In [ ]:
# cheaseBS reshapes the whole equilibrium, so this is the primary view.
fig = rh.plot_gfiles(rows_129038, camp, 129038)

In [ ]:
# Pedestal zoom -- where the axes act. Secondary: a windowed view cannot
# show distortion outside the window.
fig = rh.plot_gfiles(rows_129038, camp, 129038, rho_tor=[0.7, 1.0])

## All four together

In [ ]:
import pandas as pd
allrows = [(s, r) for s, rs in [(132543, rows_132543), (132588, rows_132588),
                                (129015, rows_129015), (129038, rows_129038)]
           for r in rs]
pd.DataFrame([{"shot": s, "axis": r["axis"], "scale": r["scale"],
               "iters": r.get("iterations"), "accepted": r.get("accepted"),
               "wall_s": round(r.get("wall_s", float("nan")))} for s, r in allrows]
             ).pivot_table(index=["shot", "axis"], columns="scale",
                           values="iters", dropna=False)